In [1]:
# Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re 
from transformers import pipeline
from datasets import Dataset
from medquad_filter import filter_medquad_data

In [2]:
# Load in the medquad dataset 
medquad_df = pd.read_csv('/kaggle/input/datasets/rpatel29/medquad/medquad.csv')
medquad_df.head()

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [3]:
# Get the number of rows in dataset 
medquad_df.shape[0]

16412

In [4]:
# Get any info regarding the medquad dataset
medquad_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16412 entries, 0 to 16411
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    16412 non-null  object
 1   answer      16407 non-null  object
 2   source      16412 non-null  object
 3   focus_area  16398 non-null  object
dtypes: object(4)
memory usage: 513.0+ KB


In [5]:
# Describe the medquad dataset
medquad_df.describe()

,question,answer,source,focus_area
count,16412,16407,16412,16398
unique,14984,15817,9,5126
top,What causes Causes of Diabetes ?,This condition is inherited in an autosomal re...,GHR,Breast Cancer
freq,20,348,5430,53


In [6]:
# Remove source and focus_area columns from medquad dataset
medquad_df.drop(columns=['source', 'focus_area'], inplace=True)
medquad_df.head()

,question,answer
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea..."
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ..."
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...


In [7]:
# Check for missing values
medquad_df.isnull().sum()

question    0
answer      5
dtype: int64

In [8]:
# Remove all rows with missing values
medquad_df.dropna(inplace=True)
medquad_df.shape[0]

16407

In [9]:
# Lowercase all question and answer text
medquad_df['question'] = medquad_df['question'].str.lower()
medquad_df['answer'] = medquad_df['answer'].str.lower()
medquad_df.head()

,question,answer
0,what is (are) glaucoma ?,glaucoma is a group of diseases that can damag...
1,what causes glaucoma ?,"nearly 2.7 million people have glaucoma, a lea..."
2,what are the symptoms of glaucoma ?,symptoms of glaucoma glaucoma can develop in ...
3,what are the treatments for glaucoma ?,"although open-angle glaucoma cannot be cured, ..."
4,what is (are) glaucoma ?,glaucoma is a group of diseases that can damag...


In [10]:
# Check for duplicates
medquad_df.duplicated().sum()

np.int64(48)

In [11]:
# Remove all duplicate rows
medquad_df.drop_duplicates(inplace=True)
medquad_df.shape[0]

16359

In [12]:
BROCHURE_REGEX = re.compile(
    r"(https?://|www\.)"
    r"|(\bfor (more|additional) information\b)"
    r"|(\blearn more\b|\bfind out more\b|\bread more\b)"
    r"|(\b(see|visit)\s+(this|the|our)\b)"
    r"|(\bsee\s+(this\s+)?(graphic|chart|table|figure)\b)"
    r"|(\bas shown in\b)"
    r"|(\bglossary\b|\bfact sheet\b|\bbrochure\b|\bhandbook\b)"
    r"|(\bcall\s+1-800\b|\btty users should call\b|\b1-800-medicare\b)"
    r"|(\bon this page\b|\bthis page\b|\bthis website\b|\bthis section\b)",
    re.IGNORECASE
)
brochure_mask = medquad_df['answer'].str.contains(BROCHURE_REGEX, na=False)
brochure_count = brochure_mask.sum()
print(f"Number of rows with brochure wording: {brochure_count}")

/tmp/ipykernel_55/2643137638.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  brochure_mask = medquad_df['answer'].str.contains(BROCHURE_REGEX, na=False)


Number of rows with brochure wording: 974


In [13]:
def brochure_match_count(text):
    return len(BROCHURE_REGEX.findall(text))

def match_position(text):
    matches = list(BROCHURE_REGEX.finditer(text))

    if not matches: 
        return None 
    
    start = matches[0].start()

    return start / len(text) 

def classify(row):

    count = row['match_count']
    position = row['match_position']
    length = row['length']

    if count >= 2 and (position is None or position <= 0.6):
        return 'delete'
    elif count >= 1 and (position is not None and position > 0.6):
        return 'trim'


medquad_df['match_count'] = medquad_df['answer'].apply(brochure_match_count)
medquad_df['match_position'] = medquad_df['answer'].apply(match_position)
medquad_df['length'] = medquad_df['answer'].str.len()
medquad_df['category'] = medquad_df.apply(classify, axis=1)


In [14]:
TRIM_REGEX = re.compile(
    r"("
    r"(for (more|additional) information.*)"
    r"|(\blearn more.*)"
    r"|(\bvisit (this|the|our)\b.*)"
    r"|(\bhttps?://\S+)"
    r"|(\bwww\.\S+)"
    r"|(\bcall\s+(\(?\d{3}\)?[-\s]?\d{3}[-\s]?\d{4}|1-800-[\w-]+).*)"
    r"|(\b1-800-[\w-]+.*)"
    r")$",
    re.IGNORECASE
)

def trim(txt):
    match_start = TRIM_REGEX.search(txt)
    if match_start is None:
        return txt 
    trimmed = txt[:match_start.start()].strip()

    if len(trimmed) < 0.5 * len(txt):
        return txt 

    return trimmed
    

# Remove all the bad rows 
medquad_df = medquad_df[medquad_df['category'] != 'delete']

# Trim all texts with brochure-like texts at the very end 
medquad_df['answer'] = medquad_df.apply(
    lambda row: trim(row['answer']) if row['category'] == 'trim' else row['answer'],
    axis=1
)

In [15]:
# Remove unnecessary columns 
medquad_df.drop(columns=['match_count', 'match_position', 'length', 'category'], inplace=True)

In [16]:
# Do a final filtering to remove any rows that still contains brochure-like wording
medquad_df = filter_medquad_data(medquad_df)

In [18]:
medquad_df.drop(columns=['should_drop'], inplace=True)

In [20]:
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

def is_aligned(question, answer):
    hypothesis = f"This text answers the question: {question}. {{}}"
    candidate_labels=['entailment', 'contradiction', 'neutral'] 
    result = classifier(answer, candidate_labels=candidate_labels,  hypothesis_template=hypothesis)

    label = result['labels'][0]
    score = result['scores'][0]

    return label == 'entailment' and score > 0.75

medquad_df['aligned'] = medquad_df.apply(lambda row: is_aligned(row['question'], row['answer']), axis=1)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [21]:
medquad_df.head(20)

,question,answer,aligned
1,what causes glaucoma ?,"nearly 2.7 million people have glaucoma, a lea...",True
2,what are the symptoms of glaucoma ?,symptoms of glaucoma glaucoma can develop in o...,False
5,what is (are) glaucoma ?,the optic nerve is a bundle of more than 1 mil...,False
6,what is (are) glaucoma ?,open-angle glaucoma is the most common form of...,False
7,who is at risk for glaucoma? ?,anyone can develop glaucoma. some people are a...,True
8,how to prevent glaucoma ?,"at this time, we do not know how to prevent gl...",True
9,what are the symptoms of glaucoma ?,"at first, open-angle glaucoma has no symptoms....",True
10,what are the treatments for glaucoma ?,"yes. immediate treatment for early stage, open...",True
11,what research (or clinical trials) is being do...,through studies in the laboratory and with pat...,True
12,who is at risk for glaucoma? ?,encourage them to have a comprehensive dilated...,True


In [22]:
medquad_df[medquad_df['aligned'] == False]

,question,answer,aligned
2,what are the symptoms of glaucoma ?,symptoms of glaucoma glaucoma can develop in o...,False
5,what is (are) glaucoma ?,the optic nerve is a bundle of more than 1 mil...,False
6,what is (are) glaucoma ?,open-angle glaucoma is the most common form of...,False
14,what is (are) high blood pressure ?,high blood pressure is a common disease in whi...,False
18,what are the symptoms of high blood pressure ?,"high blood pressure is often called the ""silen...",False
...,...,...,...
16398,what to do for cyclic vomiting syndrome ?,during the prodrome and vomiting phases of cyc...,False
16402,what are the symptoms of diabetic neuropathies...,symptoms depend on the type of neuropathy and ...,False
16404,what is (are) diabetic neuropathies: the nerve...,"peripheral neuropathy, also called distal symm...",False
16407,what is (are) diabetic neuropathies: the nerve...,focal neuropathy appears suddenly and affects ...,False


In [23]:
medquad_df = medquad_df[medquad_df['aligned'] == True]
medquad_df.drop(columns=['aligned'], inplace=True)

In [25]:
medquad_df.to_csv("medquad_cleaned.csv", index=False)